# Analysis de Datos

### Imports

In [1]:
import os
import json
from pathlib import Path
import requests
import pandas as pd
from sqlalchemy import create_engine, text

# paths
CSV_PATH = "/home/jovyan/datos/sensores.csv"
SQLITE_DIR = Path("/home/jovyan/work/data")
SQLITE_DIR.mkdir(parents=True, exist_ok=True)

PG_URI = "postgresql+psycopg2://smartcampus:smartcampus_dev@db:5432/smartcampus"
SQLITE_URI = f"sqlite:///{SQLITE_DIR / 'smartcampus.db'}"

### read Data, check structure and prepare for storage

In [2]:
df = pd.read_csv(CSV_PATH, parse_dates=["fecha_hora"])

In [3]:
# 1. Dimensions check (192 rows, 5 columns)
assert len(df) == 192, f"Expected 192 rows, got: {len(df)}"
assert set(df.columns) == {"edificio", "fecha_hora", "temperatura", "humedad", "co2_ppm"}, "Column schema mismatch."

# 2. Null values and duplicates check
assert df.isna().sum().sum() == 0, "Dataset contains null values."
assert df.duplicated().sum() == 0, "Dataset contains duplicate records."

# 3. Physical boundaries validation
assert df["temperatura"].between(-20, 60).all(), "Temperature values out of physical bounds."
assert df["humedad"].between(0, 100).all(), "Relative humidity values out of range [0, 100]."
assert (df["co2_ppm"] >= 0).all(), "Negative CO2 readings detected."

# Explicit type enforcement for persistence
df["edificio"] = df["edificio"].astype(str)
df["temperatura"] = df["temperatura"].astype(float)
df["humedad"] = df["humedad"].astype(float)
df["co2_ppm"] = df["co2_ppm"].astype(int)

print("Data quality checks completed successfully.")

Data quality checks completed successfully.


In [4]:
pg_engine = create_engine(PG_URI)

# Idempotent table truncation within an atomic transaction
with pg_engine.begin() as conn:
    conn.execute(text("DELETE FROM lecturas;"))

# Append data
df.to_sql("lecturas", con=pg_engine, if_exists="append", index=False)

# Verify loaded row count
with pg_engine.connect() as conn:
    pg_count = conn.execute(text("SELECT COUNT(*) FROM lecturas")).scalar()

assert pg_count == 192, f"Incorrect row count in PostgreSQL: {pg_count} != 192"
print(f"Records successfully loaded into PostgreSQL: {pg_count}")

Records successfully loaded into PostgreSQL: 192


In [5]:
# A) Engine-side SQL aggregation: Average temperature (ANSI SQL)
query_temp = text("""
    SELECT 
        edificio, 
        ROUND(AVG(temperatura), 2) AS temp_media
    FROM lecturas 
    GROUP BY edificio 
    ORDER BY edificio;
""")
df_temp_pg = pd.read_sql(query_temp, con=pg_engine)
print("Average temperature per building (Engine-side SQL):")
print(df_temp_pg)

# B) In-memory DataFrame aggregation: Average CO2 via full table extraction
df_lecturas_pg = pd.read_sql_table("lecturas", con=pg_engine)
df_co2_pg = (
    df_lecturas_pg.groupby("edificio", as_index=False)["co2_ppm"]
    .mean()
    .rename(columns={"co2_ppm": "co2_medio"})
)
df_co2_pg["co2_medio"] = df_co2_pg["co2_medio"].round(1)

# Identification of building with maximum CO2
idx_max_co2_pg = df_co2_pg["co2_medio"].idxmax()
top_co2_pg_edificio = df_co2_pg.loc[idx_max_co2_pg, "edificio"]
top_co2_pg_valor = df_co2_pg.loc[idx_max_co2_pg, "co2_medio"]

print("\nAverage CO2 per building (In-memory pandas):")
print(df_co2_pg)
print(f"\nBuilding with highest mean CO2: Building {top_co2_pg_edificio} ({top_co2_pg_valor} ppm)")

Average temperature per building (Engine-side SQL):
  edificio  temp_media
0        A       22.63
1        B       23.68
2        C       24.44
3        D       23.08

Average CO2 per building (In-memory pandas):
  edificio  co2_medio
0        A      822.4
1        B     1002.9
2        C     1202.4
3        D     1349.3

Building with highest mean CO2: Building D (1349.3 ppm)


### SQL Lite

In [6]:
sqlite_engine = create_engine(SQLITE_URI)

# Clean up table in SQLite
with sqlite_engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS lecturas;"))

# Append data
df.to_sql("lecturas", con=sqlite_engine, if_exists="append", index=False)

# Verify row count
with sqlite_engine.connect() as conn:
    sqlite_count = conn.execute(text("SELECT COUNT(*) FROM lecturas")).scalar()

assert sqlite_count == 192, f"Incorrect row count in SQLite: {sqlite_count} != 192"
print(f"Records successfully loaded into SQLite: {sqlite_count}")

Records successfully loaded into SQLite: 192


### Unified function

In [7]:
def cargar_y_resumir(engine, data: pd.DataFrame) -> dict:
    """
    Vendor-agnostic data pipeline function.
    Operates without internal branching based on database dialect.
    """
    # 1. Clear table to ensure idempotent execution
    with engine.begin() as conn:
        conn.execute(text("DELETE FROM lecturas;"))
    
    # 2. Persistence
    data.to_sql("lecturas", con=engine, if_exists="append", index=False)
    
    # 3. Row count extraction via SQLAlchemy Core
    with engine.connect() as conn:
        total_rows = conn.execute(text("SELECT COUNT(*) FROM lecturas")).scalar()
    
    # 4. SQL Aggregation: Average temperature (ANSI SQL)
    sql_query = text("""
        SELECT 
            edificio, 
            ROUND(AVG(temperatura), 2) AS temp_media 
        FROM lecturas 
        GROUP BY edificio 
        ORDER BY edificio;
    """)
    resumen_temp = pd.read_sql(sql_query, con=engine)
    
    # 5. pandas Aggregation: Average CO2 via full table extraction
    df_completo = pd.read_sql_table("lecturas", con=engine)
    resumen_co2 = (
        df_completo.groupby("edificio", as_index=False)["co2_ppm"]
        .mean()
        .rename(columns={"co2_ppm": "co2_medio"})
    )
    resumen_co2["co2_medio"] = resumen_co2["co2_medio"].round(1)
    
    max_co2_idx = resumen_co2["co2_medio"].idxmax()
    edificio_max_co2 = resumen_co2.loc[max_co2_idx, "edificio"]
    valor_max_co2 = resumen_co2.loc[max_co2_idx, "co2_medio"]
    
    return {
        "total_registros": total_rows,
        "temperatura_media": resumen_temp,
        "co2_medio": resumen_co2,
        "edificio_max_co2": (edificio_max_co2, valor_max_co2)
    }

# Execute unified pipeline on both database engines
res_pg = cargar_y_resumir(pg_engine, df)
res_sqlite = cargar_y_resumir(sqlite_engine, df)

print("Unified pipeline executed successfully across both engines.")

Unified pipeline executed successfully across both engines.


### strict scalar assertions

In [8]:
assert res_pg["total_registros"] == res_sqlite["total_registros"] == 192, "Row count mismatch between engines."
assert res_pg["edificio_max_co2"] == res_sqlite["edificio_max_co2"] == ("D", 1349.3), "Max CO2 mismatch between engines."

# DataFrame comparison for CO2 results
pd.testing.assert_frame_equal(res_pg["co2_medio"], res_sqlite["co2_medio"])

print("Comparative Temperature Results:")
print("--- PostgreSQL ---")
print(res_pg["temperatura_media"])
print("--- SQLite ---")
print(res_sqlite["temperatura_media"])

print("\nComparative CO2 Results:")
print("--- PostgreSQL ---")
print(res_pg["co2_medio"])
print("--- SQLite ---")
print(res_sqlite["co2_medio"])

Comparative Temperature Results:
--- PostgreSQL ---
  edificio  temp_media
0        A       22.63
1        B       23.68
2        C       24.44
3        D       23.08
--- SQLite ---
  edificio  temp_media
0        A       22.63
1        B       23.68
2        C       24.44
3        D       23.08

Comparative CO2 Results:
--- PostgreSQL ---
  edificio  co2_medio
0        A      822.4
1        B     1002.9
2        C     1202.4
3        D     1349.3
--- SQLite ---
  edificio  co2_medio
0        A      822.4
1        B     1002.9
2        C     1202.4
3        D     1349.3


### Reto D · Prueba de integración de extremo a extremo

Los apartados anteriores han trabajado **dentro** del contenedor de Jupyter.
Este último comprueba que el sistema completo funciona: que una petición HTTP
enviada desde aquí **llega a la API**, la API **escribe en PostgreSQL**, y el
registro se puede **recuperar después con SQLAlchemy**.

```text
   Jupyter                        api                        db
  ┌────────┐   POST /lecturas   ┌────────┐   INSERT INTO   ┌──────────┐
  │requests├───────────────────►│ Flask  ├────────────────►│ lecturas │
  │        │◄───────────────────┤        │                 │          │
  └───┬────┘   HTTP 201 + JSON  └────────┘                 └────┬─────┘
      │                                                        │
      │              SELECT ... (SQLAlchemy)                    │
      └────────────────────────────────────────────────────────┘
                    comprobar que el registro está
```

**Dos detalles importantes:**

1. La API se alcanza por el **nombre del servicio**, `api`, no por `localhost`.
   Desde este contenedor, `localhost` apuntaría a Jupyter, no a la API.
2. No basta con enseñar la respuesta JSON de la API. Hay que **volver a leer la
   base de datos** y demostrar que el registro se ha persistido de verdad.


In [9]:
# Direccion de la API dentro de la red de Docker Compose.
# 'api' es el NOMBRE DEL SERVICIO, igual que 'db' lo era para PostgreSQL.
API_URL = "http://api:5000"

# Comprobacion previa: la API responde y ademas puede acceder a la base de datos.
respuesta_health = requests.get(f"{API_URL}/health", timeout=10)
print(f"GET /health -> HTTP {respuesta_health.status_code}")
print(respuesta_health.json())

assert respuesta_health.status_code == 200, "La API no responde correctamente"


GET /health -> HTTP 200
{'database': 'ok', 'status': 'ok'}


In [10]:
# 1) Valor de referencia ANTES de la prueba.
#    Se cuenta con SQLAlchemy, no a traves de la API, para que la comprobacion
#    sea independiente del servicio que estamos probando.
with pg_engine.connect() as conn:
    filas_antes = conn.execute(text("SELECT COUNT(*) FROM lecturas")).scalar()

print(f"Registros en PostgreSQL antes de la prueba: {filas_antes}")


Registros en PostgreSQL antes de la prueba: 192


In [11]:
# 2) Medicion nueva y RECONOCIBLE.
#    La fecha se elige fuera del periodo del CSV (que termina el 14-10-2026 a
#    las 19:45) para que no se confunda con ninguna lectura original.
nueva_lectura = {
    "edificio": "D",
    "fecha_hora": "2026-10-15 03:30:00",
    "temperatura": 19.9,
    "humedad": 62.5,
    "co2_ppm": 1450,
}

print("Cuerpo de la peticion POST /lecturas:")
print(json.dumps(nueva_lectura, indent=2, ensure_ascii=False))


Cuerpo de la peticion POST /lecturas:
{
  "edificio": "D",
  "fecha_hora": "2026-10-15 03:30:00",
  "temperatura": 19.9,
  "humedad": 62.5,
  "co2_ppm": 1450
}


In [12]:
# 3) Envio de la peticion POST a la API.
respuesta = requests.post(
    f"{API_URL}/lecturas",
    json=nueva_lectura,
    headers={"Content-Type": "application/json"},
    timeout=10,
)

print(f"POST /lecturas -> HTTP {respuesta.status_code}")
print(json.dumps(respuesta.json(), indent=2, ensure_ascii=False))

# El enunciado exige HTTP 201 cuando la lectura se ha registrado correctamente.
assert respuesta.status_code == 201, (
    f"Se esperaba HTTP 201 y se recibio {respuesta.status_code}"
)

id_creado = respuesta.json()["lectura"]["id"]
print(f"\nLa API confirma la creacion de la lectura con id = {id_creado}")


POST /lecturas -> HTTP 201
{
  "lectura": {
    "co2_ppm": 1450,
    "edificio": "D",
    "fecha_hora": "2026-10-15 03:30:00",
    "humedad": 62.5,
    "id": 4616,
    "temperatura": 19.9
  },
  "mensaje": "Lectura almacenada correctamente."
}

La API confirma la creacion de la lectura con id = 4616


In [13]:
# 4) Comprobacion de la PERSISTENCIA REAL, con SQLAlchemy.
#    Se vuelve a consultar la base de datos y se busca esa lectura concreta por
#    su id. Esto es lo que demuestra la integracion: no basta con el JSON que
#    devuelve la API.
with pg_engine.connect() as conn:
    filas_despues = conn.execute(text("SELECT COUNT(*) FROM lecturas")).scalar()

    registro = conn.execute(
        text("""
            SELECT id, edificio, fecha_hora, temperatura, humedad, co2_ppm
            FROM lecturas
            WHERE id = :id
        """),
        {"id": id_creado},
    ).fetchone()

print(f"Registros en PostgreSQL antes : {filas_antes}")
print(f"Registros en PostgreSQL despues: {filas_despues}")
print()
print(f"Lectura recuperada de la base de datos (id={id_creado}):")
print(f"  edificio    : {registro[1]}")
print(f"  fecha_hora  : {registro[2]}")
print(f"  temperatura : {registro[3]}")
print(f"  humedad     : {registro[4]}")
print(f"  co2_ppm     : {registro[5]}")

# La lectura debe existir y los valores deben coincidir con los enviados.
assert registro is not None, "La lectura NO se ha persistido en PostgreSQL"
assert filas_despues == filas_antes + 1, (
    f"Se esperaba una fila mas: {filas_antes} -> {filas_despues}"
)
assert registro[1] == nueva_lectura["edificio"]
assert registro[5] == nueva_lectura["co2_ppm"]

print()
print("OK: la peticion POST ha quedado persistida en PostgreSQL.")


Registros en PostgreSQL antes : 192
Registros en PostgreSQL despues: 193

Lectura recuperada de la base de datos (id=4616):
  edificio    : D
  fecha_hora  : 2026-10-15 03:30:00
  temperatura : 19.90
  humedad     : 62.50
  co2_ppm     : 1450

OK: la peticion POST ha quedado persistida en PostgreSQL.


In [14]:
# 5) Resumen de la prueba de integracion, con las tres evidencias juntas.
with pg_engine.connect() as conn:
    filas = pd.read_sql(
        text(f"SELECT * FROM lecturas WHERE id = {id_creado}"), conn
    )

print("=== EVIDENCIA DE LA PRUEBA DE EXTREMO A EXTREMO ===")
print(f"1) Peticion enviada  : POST {API_URL}/lecturas")
print(f"2) Respuesta recibida: HTTP {respuesta.status_code}")
print(f"3) Registro en la BBDD recuperado con SQLAlchemy:\n")
filas


=== EVIDENCIA DE LA PRUEBA DE EXTREMO A EXTREMO ===
1) Peticion enviada  : POST http://api:5000/lecturas
2) Respuesta recibida: HTTP 201
3) Registro en la BBDD recuperado con SQLAlchemy:



,id,edificio,fecha_hora,temperatura,humedad,co2_ppm
0,4616,D,2026-10-15 03:30:00,19.9,62.5,1450
